<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor: experiment notebook

This notebook compares five formulations for the periodic two-well **committor probability** PDE, following the methods in [Tepakbong et al. (2026)](https://arxiv.org/abs/2607.19167).

**Workflow:** read the equation; run Setup/tests and the tiny preflight on CPU; prepare the finite-difference reference and Monte Carlo labels on CPU; then use the configured device for training. The default uses GPU; choosing `device: cpu` keeps the entire run on CPU. The longer CPU preparation reports progress and saves to one dated Drive folder. No GPU is requested at startup.

[Experiment guide: full derivation, loss definitions, saved-file map, and references](https://github.com/n2pakbong/boundary-adapted-pinns/blob/c243c6e23e3bc450d347947ca173045bc91aa2d1/experiments/committor/periodic_two_well/README.md).

## 1. The mathematical problem

Consider overdamped Langevin dynamics on the flat torus $\mathbb T^2=[0,1)^2$:

$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t \pmod 1,
\qquad
V(x_1,x_2)=a\left[1-\cos\!\left(4\pi(x_1-\tfrac14)\right)\right]
+b\left[1-\cos(2\pi x_2)\right].
$$

Let $A$ and $B$ be two disjoint periodic disks, centered at $(1/4,0)$ and $(3/4,0)$. The **forward committor** is

$$
q(x)=\mathbb P_x(\tau_B<\tau_A),
$$

the probability that a trajectory started at $x$ reaches $B$ before $A$. It solves

$$
\mathcal L q
=-\nabla V\cdot\nabla q+\varepsilon\Delta q=0
\quad\text{in }D=\mathbb T^2\setminus(A\cup B),
$$

with $q=0$ on $\partial A$ and $q=1$ on $\partial B$. Thus this notebook studies a hitting probability, not a mean escape time.

The default experiment uses $a=1$, $b=0.6$, $\varepsilon=0.4$, and periodic chordal radius $r=0.5$. The finite-difference solution is a numerical target for error measurement; it is not exact ground truth.

## 2. What runs, and what uses compute

| Stage | Work | Compute |
|---|---|---|
| Setup | Install code and run tests | CPU |
| Preflight | Tiny one-epoch run of all five methods | CPU |
| Preparation | Finite-difference references and shared Monte Carlo labels | CPU; saved to Drive |
| Training | Five networks and solution-error evaluation | Configured CPU or GPU |
| Inspect/evaluate | Saved curves, losses, errors; checkpoint evaluation | CPU |

Leave Colab on CPU through preparation. Training uses a GPU only when the saved configuration selects `device: cuda`; `device: cpu` keeps the full run on CPU. Settings are in `experiments/committor/periodic_two_well/configs/gpu_pilot.json`; this single-seed comparison is exploratory, not a final paper result.

## 3. Setup and CPU tests

Keep Colab on CPU and run the next cell. Leave the branch name blank to use `main`; choose a branch only when you deliberately need a different code version. Setup installs the code and runs CPU tests; it does not train.

After CPU preparation, use the device selected in the saved configuration: switch to GPU and rerun Setup if it says `cuda`, or stay on CPU if it says `cpu`. GPU Setup skips CPU tests; the training cell verifies that saved preparation matches the installed revision.

In [ ]:
# CPU setup: select a Git branch, install it, and run mathematical and tiny training tests.
from pathlib import Path
import base64
import os
import subprocess

from google.colab import userdata

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Show output and offer the complete log as a download if the command fails."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        print(f"\nCommand failed (exit {return_code}). The full traceback is above.")
        try:
            from google.colab import files
            files.download(str(log_path))
            print("Attach the downloaded console.log if you need help diagnosing the error.")
        except Exception as download_error:
            print(f"Log download did not start: {download_error}. Copy the traceback above.")
        raise RuntimeError("Command failed. The traceback and downloadable log are above.")

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = input('Git branch to install [main]: ').strip() or 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Publicly readable repositories can be cloned without authentication.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch the selected branch. Check the branch name and repository access, then rerun Setup.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import torch
if torch.cuda.is_available():
    print('GPU runtime detected. Installed the selected branch; CPU tests are not rerun here.')
    print('They must have passed in the CPU runtime before the prepared experiment folder was created.')
else:
    subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
    print('Setup and CPU tests passed. Code revision:', revision)


## 4. CPU preflight: check the complete pipeline without GPU

Run this after Setup passes. It executes all five methods for one epoch with tiny networks, one fixed Monte Carlo anchor, eight short paths, and coarse finite-difference grids. This is a wiring check, not a useful accuracy result.

Its outputs are written under `/content/periodic_committor_preflight`, which is temporary Colab storage. It does not clutter Drive. The cell sets a flag only after it completes successfully; the later training cell refuses to start unless this flag is set in the current runtime.

In [ ]:
# CPU only: run the full pipeline in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The full CPU experiment preparation is now available.')

## 5. Boundary conditions and the five trial functions

For each boundary set $j\in\{A,B\}$, the code builds a smooth level-set function $\omega_j$ and a normalized distance-like function $d_j$:

$$
\omega_j(x)=4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)-r^2,
\qquad
d_j(x)=\frac{\omega_j(x)}
{\sqrt{\omega_j(x)^2+\lvert\nabla\omega_j(x)\rvert^2}}.
$$

Here $c_j$ is the center of set $j$. The boundary is where $\omega_j=0$; in the exterior domain, both $d_A$ and $d_B$ are positive. Define the smooth lift $g$ and the two-boundary factor $\rho$ by

$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},
\qquad
\rho(x)=\frac{d_A(x)d_B(x)}
{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$

Then $g=0$ on $\partial A$, $g=1$ on $\partial B$, and $\rho=0$ on both boundaries. The three exact-boundary trial functions use

$$
q_\theta(x)=g(x)+\rho(x)^\alpha N_\theta(x).
$$

Boundary-adapted uses $\alpha=1$, over-enforced uses $\alpha=3/2$, and under-enforced uses $\alpha=1/2$. Each satisfies both Dirichlet values by construction. The network output is periodic because the network receives sine and cosine coordinates. For background on the standard PINN residual and exact distance-function boundary enforcement, see [Raissi et al. (2019)](https://doi.org/10.1016/j.jcp.2018.10.045) and [Sukumar and Srivastava (2022)](https://doi.org/10.1016/j.cma.2021.114333); the exponent variants listed here are the specific cases in this experiment.

## 6. Which loss belongs to which method?

Every method uses the same interior collocation points and the same Monte Carlo anchor labels within a comparison. Write the interior residual loss and shared solution-value data loss as

$$
\widehat J_{\rm pde}(\theta)
=\frac{1}{N_D}\sum_{i=1}^{N_D}
\left|\mathcal Lq_\theta(x_i)\right|^2,
\qquad
\widehat J_{\rm data}(\theta)
=\frac{1}{N_{\rm data}}\sum_{i=1}^{N_{\rm data}}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$

| Method | Trial function | Trained loss components | Boundary treatment |
|---|---|---|---|
| Standard PINN (`simple_pinn`) | $q_\theta=N_\theta$ | PDE, Monte Carlo data, ordinary boundary residual | Samples both boundaries and penalizes the Dirichlet residual. |
| Boundary-adapted | $g+\rho N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Over-enforced | $g+\rho^{3/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Under-enforced | $g+\rho^{1/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Variational/Nitsche | $q_\theta=N_\theta$ | Interior-plus-flux loss + Nitsche boundary loss + Monte Carlo data | Weakly imposes the boundary values using Nitsche's method. |

Only the standard PINN has the ordinary boundary-residual component:

$$
\widehat J_{\rm bc}
=\frac{1}{N_A}\sum_i q_\theta(x_i^A)^2
+\frac{1}{N_B}\sum_i\bigl(q_\theta(x_i^B)-1\bigr)^2.
$$

The boundary-adapted, over-enforced, and under-enforced methods have **no extra boundary penalty**. The variational method's stabilization term belongs to the Nitsche weak formulation; it is not applied to an exact-boundary ansatz. The adaptive objective and exact component names are defined in Section 8; the variational method is derived in Section 9. The main experiment guide links each equation to its implementation.

## 7. Monte Carlo labels: the slow CPU stage and its live progress

The committor also has a stochastic representation. Starting at anchor $z_i$, independent Euler–Maruyama trajectories approximate the Langevin dynamics:

$$
X_{n+1}
=\left[X_n-\nabla V(X_n)\Delta t
+\sqrt{2\varepsilon\Delta t}\,\xi_n\right]\pmod 1,
\qquad \xi_n\sim\mathcal N(0,I_2).
$$

This carries over the pointwise solution-value regularizer used in the mean-escape-time notebooks: here the target value is the committor probability, and it complements rather than replaces the PDE residual. Each trajectory stops on first entering $A$ or $B$, or when it reaches the configured step cap. The label is the fraction of resolved trajectories that hit $B$ first:

$$
\widehat q_i
=\frac{\#\{\text{resolved trajectories from }z_i\text{ that hit }B\text{ first}\}}
{\#\{\text{resolved trajectories from }z_i\}}.
$$

Capped, unresolved paths are **not** counted as hits of $A$. Training stops with an error if an anchor's resolved fraction is below the configured minimum. The reported binomial standard error is conditional on the resolved paths; if paths remain unresolved, the guide explains the possible censoring bias and time-step error.

**What you will see while it runs:** reports show the step cap, paths resolved to (A) and (B), paths still active, anchors meeting the configured resolution threshold, the minimum and mean resolved fractions, and elapsed time. They appear at the configured fraction of the step cap or after the configured time interval, whichever comes first. The final resolution summary appears before training. After generation, the next cell saves and displays a plot of anchor locations colored by their estimated committor values, plus each label with its estimated Monte Carlo standard error. All methods share the same anchors and labels in `monte_carlo_labels.npz` and `monte_carlo_labels.json`. Neural-network approaches to committor computation are discussed, for example, by [Khoo, Lu, and Ying (2018)](https://arxiv.org/abs/1802.10275) and [Song, Cameron, and Yang (2023)](https://arxiv.org/abs/2306.12268).

## 8. Adaptive weighting in the paper's notation

The paper denotes the total adaptive objective by $\widehat J$ and its active empirical loss components by $\widehat J_i$. For each component, $\lambda_i>0$ is a trainable scale. Its objective (Eq. (33)) is

$$
\widehat J(\theta,\{\lambda_i\}_{i\in\mathcal I})
=\sum_{i\in\mathcal I}\frac{1}{2\lambda_i^2}\widehat J_i(\theta)
+\sum_{i\in\mathcal I}\log\lambda_i.
$$

See [Tepakbong et al. (2026), Eq. (33)](https://arxiv.org/abs/2607.19167). The active set $\mathcal I$ follows the method being trained:

- Standard PINN (simple_pinn): $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm bc}$ is the sampled Dirichlet residual.
- Boundary-adapted, over-enforced, and under-enforced: $\{\mathrm{pde},\mathrm{data}\}$. Their boundary values are built into the ansatz, so there is no boundary-residual term.
- Variational/Nitsche: $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm pde}$ is the squared interior-plus-flux residual, and $\widehat J_{\rm bc}$ is the squared Nitsche boundary contribution (symmetric consistency plus stabilization). This is not the standard PINN's pointwise boundary residual.

For this committor, $\widehat J_{\rm pde}$ uses the residual $\mathcal Lq_\theta=0$, and $\widehat J_{\rm data}$ compares $q_\theta(z_i)$ with the simulated committor label $\widehat q_i$. The loss definitions are in Section 6 and the Nitsche split is derived in Section 9. In the training logs, the three optimized Nitsche components are pde_loss, boundary_loss, and data_loss. The separate Nitsche term mean squares and the full weak-residual mean square are diagnostics, not separately weighted losses.

**How the code's saved scale corresponds to $\lambda_i$.** The code stores each trainable scale in log form, $s_i=\log\sigma_i$, and multiplies each raw loss by $e^{-2s_i}$. The paper's parameter is $\lambda_i=e^{s_i}/\sqrt{2}$, so $e^{-2s_i}=1/(2\lambda_i^2)$. Thus each weighted component has exactly the paper's coefficient. The code's total objective differs from the paper's displayed objective only by the constant $|\mathcal I|\log(2)/2$, which has zero gradient and does not change training. The raw loss components and effective weights are logged separately. The trainable-scale construction follows [Kendall, Gal, and Cipolla (2018)](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).

The notebook's `training.log_interval_seconds` setting controls the wall-clock interval between live console reports. The first and final epochs always print; each report includes the epoch, elapsed time, ETA, objective, and active raw loss components. Regardless of that interval, every epoch is written to `metrics.jsonl` with raw components, weighted components, effective weights, learning rate, and timing. Thus `losses.png` shows the full per-epoch history. `training.checkpoint_every` separately controls restartable checkpoints. Final raw loss components, the best epoch, and the objective are saved in `final_training_metrics.json` and `summary.json`.

Each trainable log scale starts at `training.adaptive_loss_log_scale_init` (zero in the pilot), then is optimized jointly with the network using the configured optimizer and learning-rate schedule. A zero starting value makes every raw loss coefficient initially one in the code objective.


## 9. Variational/Nitsche method: how its two trained losses are formed

Because the drift is a gradient, set $m(x)=e^{-V(x)/\varepsilon}$. The strong committor equation is equivalent to

$$
\mathcal Lq=0
\quad\Longleftrightarrow\quad
\nabla\cdot\left(\varepsilon m\nabla q\right)=0.
$$

For periodic Gaussian test functions $\phi_j$ centered at $c_j$, let $n_D$ be the outward normal of the exterior domain $D$, pointing into the removed sets:

$$
\phi_j(x)=\exp\!\left[
-\frac{4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)}{2\ell^2}
\right].
$$

The interior-plus-flux contribution is

$$
A_j=
\varepsilon\int_D m\nabla q_\theta\cdot\nabla\phi_j\,dx
-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds.
$$

The symmetric consistency term and Nitsche stabilization moment are

$$
C_j=-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds,
\qquad
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds,
$$

where $g=0$ on $\partial A$ and $g=1$ on $\partial B$. Together they form the Nitsche boundary contribution $B_j=C_j+P_j$. The full weak residual is $R_j=A_j+B_j$.

The paper's VPINN loss split is

$$
\widehat J_{\rm pde}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}A_j^2,
\qquad
\widehat J_{\rm bc}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}B_j^2.
$$

These two losses are optimized with the common committor data term $\widehat J_{\rm data}$. In the training log they are named pde_loss, boundary_loss, and data_loss. Nitsche stabilization belongs inside the boundary component; it is not an extra loss on an exact-boundary ansatz or the standard PINN's pointwise boundary residual. The separate volume, flux, consistency, stabilization, and full-residual mean squares are diagnostics, not additional adaptive losses. This follows [Liao and Ming (2021)](https://doi.org/10.4208/cicp.OA-2020-0219) and [Berrone et al. (2023)](https://doi.org/10.1016/j.heliyon.2023.e18820).

**Check the equations in code:** the PDE and boundary geometry are in src/boundary_pinn/problems/committor/periodic_two_well.py; Monte Carlo and weak-form integrals are in paper_methods.py; ansatzes, active loss sets, adaptive scales, and epoch logs are in training.py; orchestration and saved error tables are in comparison.py.

## 10. Experiment settings: what you control

The tracked preset `experiments/committor/periodic_two_well/configs/gpu_pilot.json` reproduces the successful five-method run. The next cell loads it and lets you override settings for a new run. Every setting used by the run is printed and saved as `comparison_config.json` in that run's Drive folder. After Colab restarts, training reads that saved copy; it does not reload or silently change your choices.

### Why these defaults were used

The five methods share the same periodic MLP, initialization seed, collocation budget, Monte Carlo anchors and labels, optimizer, and 6,000-epoch budget where applicable. This gives a controlled comparison of the boundary ansatz and weak formulation.

The network maps the torus coordinates to $(\\sin(2\\pi x_1),\\sin(2\\pi x_2),\\cos(2\\pi x_1),\\cos(2\\pi x_2))$, then applies a fully connected MLP with width 64 and depth 4. `tanh` is smooth for the strong residual's second derivatives. The pilot uses Xavier-normal initialization, gain 1, zero biases, and seed 7. These are baseline choices, not a tuned architecture. A second option, `periodic_fourier_mlp`, uses the integer harmonics in `model.frequencies` (for example, $[1,2,4]$); higher frequencies add representational detail but can make PDE derivatives larger.

Training uses Adam at learning rate $10^{-3}$, with exponential decay factor $0.9999$ each epoch, for 6,000 epochs per method. The 1,024 interior PDE points are resampled each epoch; the standard PINN also samples 256 points from each boundary. Nitsche has separate volume, test-function, and boundary quadrature counts.

The pointwise committor data term uses 32 anchor locations and 256 simulated trajectories per anchor (8,192 paths total). The label noise is Monte Carlo sampling variation in the hit fraction; there is no separate additive-noise setting. Increasing `paths_per_anchor` reduces that sampling variation. The time step and path cap change the trajectory approximation and censoring behavior.

### Settings you can vary

| Part of the experiment | Preset | Setting to edit | Meaning |
|---|---:|---|---|
| Methods | all five | `methods` | Select one or more named formulations. |
| Problem | $a=1$, $b=0.6$, $\varepsilon=0.4$; centers $(0.25,0)$ and $(0.75,0)$; radius $0.5$ | `problem` | Edit drift-potential parameters, source/target centers, and target-set radius. |
| Hardware/precision | CUDA; float64 | `device`, `dtype` | Choose CPU or CUDA training and float32 or float64. Full CPU training avoids GPU allocation but can be slower. |
| Network | periodic MLP; width 64; depth 4; `tanh` | `model.architecture`, `width`, `depth`, `activation` | Choose `periodic_mlp` or `periodic_fourier_mlp`. The latter also uses `model.frequencies`. Activations: `tanh`, `relu`, `gelu`, `silu`, `softplus`. |
| Initialization | seed 7; Xavier normal; gain 1; zero biases | `seed`, `model.initialization`, `model.gain` | Supported weight initializers: `xavier_normal`, `xavier_uniform`. |
| Adaptive loss weights | initial code log scale 0 | `training.adaptive_loss_log_scale_init` | Initial value for each trainable loss log scale; all methods then learn their weights during optimization. |
| Training budget | 6,000 epochs | `training.epochs` | Number of optimizer updates per selected method; no early stopping. |
| PDE points | 1,024 per epoch | `training.collocation_points` | Fresh interior points sampled each epoch. |
| Standard PINN boundary points | 256 per boundary | `training.boundary_points` | Used only for the standard PINN's pointwise Dirichlet residual. |
| Optimizer | Adam; learning rate $10^{-3}$ | `training.optimizer`, `learning_rate` | Adam and AdamW are supported; betas, epsilon, weight decay, and AMSGrad are configurable. |
| Learning-rate schedule | exponential, $\gamma=0.9999$ per epoch | `training.scheduler` | Choose constant, exponential, or step decay. |
| Console report/checkpoints | every 30 seconds / every 100 epochs | `training.log_interval_seconds`, `checkpoint_every` | Console status is time-based; a complete metric row is saved every epoch for the full loss curves. Checkpoints have their own epoch interval. |
| Fidelity data | 32 anchors × 256 paths | `training.monte_carlo` | Change anchor count, paths per anchor, seed, time step, step cap, resolution threshold, and progress cadence. |
| Nitsche quadrature | 1,024 volume; 32 tests; 512 boundary | `training.nitsche` | Quadrature and test-function settings used only by the variational method. |
| Reference/evaluation | grids 128 and 256 | `reference_grid_sizes`, `evaluation_batch_size` | Controls reference resolution and evaluation memory use. |

Edit the comments in the next cell to choose a different setting. Leave an override out to keep its preset value. These choices apply to CPU preparation and the later training stage; the prepared folder saves the exact configuration. A Colab `/content` copy disappears when that runtime is deleted, but the run's `comparison_config.json` is saved in Drive. For a reusable, version-controlled preset, copy that file into `experiments/committor/periodic_two_well/configs/` under a descriptive filename and set `PRESET_CONFIG_PATH` in the next cell to that filename. The full parameter definitions and defaults are in the [configuration guide](https://github.com/n2pakbong/boundary-adapted-pinns/blob/main/experiments/committor/periodic_two_well/configs/README.md).


In [ ]:
# Load the version-controlled preset. Add only the settings you want to change below.
from pathlib import Path
import json
from pprint import pprint

PRESET_CONFIG_PATH = Path(
    "experiments/committor/periodic_two_well/configs/gpu_pilot.json"
)
RUN_CONFIG = json.loads(PRESET_CONFIG_PATH.read_text(encoding="utf-8"))

RUN_OVERRIDES = {
    # Examples: uncomment a line or block, then change its value.
    # "seed": 8,
    # "problem": {
    #     "a": 1.0,
    #     "b": 0.6,
    #     "epsilon": 0.4,
    #     "set_radius": 0.5,
    # },
    # "methods": ["boundary_adapted", "variational_nitsche"],
    # "model": {
    #     "architecture": "periodic_fourier_mlp",
    #     "frequencies": [1, 2, 4],
    #     "width": 96,
    #     "depth": 5,
    #     "activation": "gelu",
    #     "initialization": "xavier_uniform",
    #     "gain": 1.0,
    # },
    # "training": {
    #     "epochs": 8000,
    #     "collocation_points": 2048,
    #     "learning_rate": 0.0005,
    #     "adaptive_loss_log_scale_init": 0.0,
    #     "optimizer": {
    #         "name": "adamw",
    #         "betas": [0.9, 0.999],
    #         "eps": 1e-8,
    #         "weight_decay": 0.0001,
    #     },
    #     "scheduler": {"name": "exponential", "gamma": 0.9999},
    #     "log_interval_seconds": 60,
    #     "checkpoint_every": 200,
    #     "monte_carlo": {
    #         "n_anchors": 48,
    #         "paths_per_anchor": 512,
    #         "dt": 0.001,
    #         "max_steps": 100000,
    #         "seed": 1007,
    #         "minimum_resolved_fraction": 0.9,
    #         "progress_interval_fraction": 0.01,
    #         "progress_interval_seconds": 10,
    #     },
    # },
}

def merge_settings(base, changes):
    for key, value in changes.items():
        if isinstance(value, dict):
            base[key] = merge_settings(base.get(key, {}).copy(), value)
        else:
            base[key] = value
    return base

RUN_CONFIG = merge_settings(RUN_CONFIG, RUN_OVERRIDES)
RUN_CONFIG_PATH = Path("/content/periodic_committor_selected_config.json")
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + "\n", encoding="utf-8")

print("Selected experiment settings (saved copy is made during CPU preparation):")
pprint({
    "seed": RUN_CONFIG["seed"],
    "methods": RUN_CONFIG["methods"],
    "model": RUN_CONFIG["model"],
    "training": RUN_CONFIG["training"],
    "reference_grid_sizes": RUN_CONFIG["reference_grid_sizes"],
    "evaluation_batch_size": RUN_CONFIG["evaluation_batch_size"],
})
print("Temporary selected-config file:", RUN_CONFIG_PATH)


## 11. Optional: clean up old Drive folders before a new run

This is optional and does not affect the code or results. If Drive contains temporary CPU-preflight folders or incomplete run folders, the next cell lists safe cleanup candidates. Run it only if you want to remove those temporary files.

It first prints the exact candidate paths. It excludes any experiment folder containing checkpoints, references, plots, or other result files. **Nothing is deleted unless you type `DELETE` at the prompt.** Completed runs are preserved. No new preflight or experiment outputs are written to Drive by the CPU preflight; new experiment results are written to a new dated run folder.

In [ ]:
# Optional cleanup: list eligible old folders, then require an explicit confirmation.
from pathlib import Path
import re
import shutil
from google.colab import drive

drive.mount('/content/drive')
old_run_root = Path('/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor')
known_startup_files = {'console.log', 'comparison_config.json', 'comparison_metadata.json'}
cleanup_candidates = []
if old_run_root.is_dir():
    for folder in sorted(old_run_root.iterdir()):
        if not folder.is_dir():
            continue
        if re.fullmatch(r'preflight_\d{8}_\d{6}', folder.name):
            cleanup_candidates.append(folder)
        elif re.fullmatch(r'periodic_committor_\d{8}_\d{6}', folder.name):
            children = list(folder.iterdir())
            if children and all(child.is_file() and child.name in known_startup_files for child in children):
                cleanup_candidates.append(folder)

if not cleanup_candidates:
    print('No eligible old preflight or empty failed-pilot folders were found. Nothing was deleted.')
else:
    print('Eligible folders (completed runs and folders containing checkpoints/results are excluded):')
    for folder in cleanup_candidates:
        print(' -', folder)
    confirmation = input('To permanently delete exactly these listed folders, type DELETE: ').strip()
    if confirmation == 'DELETE':
        for folder in cleanup_candidates:
            shutil.rmtree(folder)
            print('Deleted:', folder)
        print(f'Deleted {len(cleanup_candidates)} listed folders.')
    else:
        print('No folders deleted.')


## 12. CPU experiment preparation: reference grids and Monte Carlo labels

After Setup and preflight, run this with a CPU runtime. It solves both finite-difference grids and generates the shared Feynman–Kac labels, with grid timings and live path progress. It does **not** train a network or use GPU.

It runs the settings selected in Section 10. The dated Drive folder contains the resolved `comparison_config.json`, code revision, reference, labels, standard errors, progress metadata, Monte Carlo anchor plot, and console log. Keep this folder for the training stage. If Drive is already mounted, continue.

The preparation records its code revision. If you change repository revision after preparation, rerun this CPU-only cell; the training stage refuses files made with a different code revision.

In [ ]:
# CPU only: prepare the reference and common labels before allocating a GPU.
from datetime import datetime
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive
from IPython.display import Image, display

if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('CPU preparation blocked: run Setup and pass the CPU preflight in this CPU runtime first.')

torch.set_num_threads(2)
if torch.cuda.is_available():
    raise RuntimeError(
        'A GPU runtime is attached. Stop here and change Runtime > Change runtime type to CPU. '
        'This potentially long preparation stage must finish before allocating a GPU.'
    )

drive.mount('/content/drive')
run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preparation folder (keep this path for training):', RUN_ROOT, flush=True)

run_logged([
    'periodic-committor',
    str(RUN_CONFIG_PATH),
    '--output-dir', RUN_ROOT,
    '--prepare-cpu-only',
], Path(RUN_ROOT) / 'console.log')
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise RuntimeError('CPU preparation ended without cpu_preparation.json; check the console output above.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
preflight_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('preparation_git_revision') != preflight_revision:
    raise RuntimeError('CPU preflight and prepared files came from different code revisions. Repeat Setup and CPU preparation from one branch revision.')
prep_record['cpu_preflight_passed'] = True
prep_record['cpu_preflight_git_revision'] = preflight_revision
prep_path.write_text(json.dumps(prep_record, indent=2) + '\n', encoding='utf-8')
print('CPU preparation is complete. No model training has run.', flush=True)
anchor_plot = Path(RUN_ROOT) / 'monte_carlo_anchors.png'
if anchor_plot.is_file():
    print('Monte Carlo anchor plot (inspect before training):', anchor_plot)
    display(Image(filename=str(anchor_plot)))

## 13. Training: use the device selected in the configuration

CPU preparation is complete and saved on Drive. The saved `device` setting chooses where training runs:

- For the preset `device: cuda`, choose **Runtime → Change runtime type → GPU**. Colab restarts; run Setup again with the same branch. Do not rerun preflight or preparation.
- If you changed `device` to `cpu`, keep Colab on CPU. This avoids allocating a GPU runtime; the full training may take much longer.

Then run the next cell and paste the Drive folder printed during preparation. It checks the preparation marker and repository revision, loads the saved reference, labels, and exact configuration, and trains the selected methods. It reports losses at the configured wall-clock interval and saves checkpoints in the same folder. It never repeats the CPU preparation.

In [ ]:
# Train only on the device saved in the prepared configuration; references and Monte Carlo labels are already on Drive.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = input('Paste the CPU preparation folder path printed in Section 12: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError(
        f'No completed CPU preparation was found in {RUN_ROOT}. '
        'Return to the CPU runtime and finish Section 12 first.'
    )
config_path = Path(RUN_ROOT, 'comparison_config.json')
if not config_path.is_file():
    raise FileNotFoundError(f'The prepared folder is missing its saved configuration: {RUN_ROOT}')
selected_config = json.loads(config_path.read_text(encoding='utf-8'))
training_device = str(selected_config['device'])
if training_device.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError(
        'This run configuration selects CUDA. Choose Runtime > Change runtime type > GPU, '
        'reconnect, run Setup with the same branch, then rerun this cell. Do not repeat CPU preparation.'
    )
if training_device == 'cpu' and torch.cuda.is_available():
    raise RuntimeError(
        'This run configuration selects CPU. Change Runtime > Change runtime type to CPU '
        'before training so a GPU runtime is not allocated.'
    )
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
if prep_record.get('cpu_preflight_passed') is not True:
    raise RuntimeError('This folder was not prepared after a passing CPU preflight. Return to the CPU runtime and complete Sections 3, 4, and 12.')
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This prepared folder belongs to a different code revision. Repeat CPU setup, preflight, and preparation using the same branch revision.')
print(f'{training_device.upper()} training outputs will be saved in:', RUN_ROOT, flush=True)
run_logged([
    'periodic-committor',
    str(config_path),
    '--output-dir', RUN_ROOT,
    '--train-prepared',
], Path(RUN_ROOT) / 'console.log', append=True)

## 14. Inspect the completed run

Run after training. In a fresh runtime, the cell asks for the Drive folder path. It reads saved files only and does not train. It displays the finite-difference grid check; absolute and relative $L^2$, $H^1$, $H^1$-seminorm, $H^2$, and $H^2$-seminorm errors; final raw loss components; and saved plots.

Errors use the finest configured finite-difference solution. Grid-to-grid differences estimate reference-resolution sensitivity, not an error bound. Figures are previews; arrays, metrics, and checkpoints are saved in the printed Drive folder.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'summary.json').is_file():
    RUN_ROOT = input('Paste the completed Drive run folder path: ').strip()
if not Path(RUN_ROOT, 'summary.json').is_file():
    raise FileNotFoundError(f'No summary.json found in {RUN_ROOT}')

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run Section 15 to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

## 15. Re-evaluate saved checkpoints on CPU

This separate cell reloads each method's `best.pt` checkpoint and recomputes errors against the saved finite-difference reference. It uses CPU only, does not retrain, and does not overwrite the original `summary.json`. It displays the Sobolev errors and final raw training-loss components and writes `saved_model_evaluation.json` to the same run folder.

If using a fresh Colab runtime, paste the experiment folder path when prompted.

In [ ]:
# CPU only: compare saved best.pt checkpoints with the FDM target; this does not train.
from pathlib import Path
import json
import subprocess
from google.colab import drive
from IPython.display import display
import pandas as pd

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'comparison_config.json').is_file():
    RUN_ROOT = input('Paste the completed experiment Drive folder path: ').strip()

if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'No comparison_config.json found in {RUN_ROOT}')

command = ['periodic-committor', '--output-dir', str(RUN_ROOT), '--evaluate-saved-models']
subprocess.run(command, check=True)

report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])

print('Final training-loss components from the saved run:')
loss_rows = {}
for method, result in report['methods'].items():
    training = result.get('final_training_metrics', {})
    loss_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **training.get('raw_loss_components', {}),
        'MC data term used': training.get('monte_carlo_labels') is not None,
    }
display(pd.DataFrame(loss_rows).T)
print('Full saved report:', report_path)

## 16. Resume an interrupted run

Use this only when training stopped before all selected methods finished. Check the saved `device`: use a GPU runtime if it says `cuda`, or a CPU runtime if it says `cpu`; then run Setup with the same code branch. Paste the saved Drive folder path. The runner resumes from each method's `last.pt`; it does not repeat preparation.

It checks the code revision and saved objective version before resuming. If either differs, it stops before loading checkpoints and tells you to start a new dated run folder.

For a new experiment, return to the training section and use a new prepared folder.

In [ ]:
# Resume only an interrupted run; the saved configuration chooses CPU or GPU.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = input('Paste the interrupted run folder path printed at the start of training: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError('That folder has no completed CPU preparation. Paste the run folder path from this notebook.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
config_path = Path(RUN_ROOT, 'comparison_config.json')
if not config_path.is_file():
    raise FileNotFoundError(f'The run folder is missing its saved configuration: {RUN_ROOT}')
selected_config = json.loads(config_path.read_text(encoding='utf-8'))
training_device = str(selected_config['device'])
if training_device.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('This saved run requires a GPU runtime. Switch to GPU, run Setup with the same branch, then rerun this cell.')
if training_device == 'cpu' and torch.cuda.is_available():
    raise RuntimeError('This saved run uses CPU. Change the runtime type to CPU before resuming.')
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_passed') is not True or prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This run folder does not match the passed CPU preflight and installed code revision.')

run_logged([
    'periodic-committor',
    str(config_path),
    '--output-dir', RUN_ROOT,
    '--train-prepared',
    '--resume',
], Path(RUN_ROOT) / 'console.log', append=True)
print(f'The interrupted {training_device.upper()} run has resumed and completed.')

## References

- Y. Liao and P. Ming, “Deep Nitsche Method: Deep Ritz Method with Essential Boundary Conditions,” *Communications in Computational Physics* 29(5) (2021), 1365–1384. [doi:10.4208/cicp.OA-2020-0219](https://doi.org/10.4208/cicp.OA-2020-0219).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- A. Kendall, Y. Gal, and R. Cipolla, “Multi-task learning using uncertainty to weigh losses for scene geometry and semantics,” *CVPR* (2018). [paper](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).